In [1]:
import torch
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torch.utils.data import random_split,Subset
import torch.nn as nn


In [2]:
initial_transform = transforms.ToTensor()

train_dataset_full = torchvision.datasets.CIFAR10(
    root="../data",
    train=True,
    transform=initial_transform,
    download=True
)

In [3]:
#setting up the mean
sum_pixels = torch.zeros(3)
total_pixels = 0

for image, label in train_dataset_full:
    sum_pixels[0] += torch.sum(image[0])
    sum_pixels[1] += torch.sum(image[1])
    sum_pixels[2] += torch.sum(image[2])
    total_pixels += image[0].numel()

mean = sum_pixels / total_pixels
print(f"Mean: {mean}")
sum_squared_diff = torch.zeros(3)
for image, label in train_dataset_full:
    difference_R = image[0]- mean[0]
    difference_G = image[1]- mean[1]
    difference_B = image[2]- mean[2]
    sum_squared_diff[0] += (difference_R**2).sum()
    sum_squared_diff[1] += (difference_G**2).sum()
    sum_squared_diff[2] += (difference_B**2).sum()

variance = sum_squared_diff / total_pixels
std = torch.sqrt(variance)
print(f"Std: {std}")




Mean: tensor([0.4914, 0.4822, 0.4465])
Std: tensor([0.2470, 0.2435, 0.2616])


In [4]:
#transform the data
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean,std)
])
eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean,std)
])

In [5]:
train_dataset_full = torchvision.datasets.CIFAR10(
    root="../data",
    train=True,
    transform= train_transform,
    download=True
)
val_dataset_full = torchvision.datasets.CIFAR10(
    root="../data",
    train=True,
    transform= eval_transform,
    download=True
)



In [6]:
#Randomly split the indices
indices = torch.randperm(len(train_dataset_full))
train_indices = indices[:45000]
val_indices = indices[45000:]

train_dataset = Subset(train_dataset_full, train_indices)
val_dataset = Subset(val_dataset_full, val_indices)

In [7]:
test_dataset = torchvision.datasets.CIFAR10(
    root="../data",
    train=False,
    transform= eval_transform,
    download=True
)

 - training set is 45000
 - validation set is 5000
  - test set is 10000

In [8]:
print(f"Training dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

Training dataset size: 45000
Validation dataset size: 5000
Test dataset size: 10000


In [9]:
#next is dataloader
train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(dataset=val_dataset, batch_size=1000, shuffle=False)
test_loader = DataLoader(dataset=test_dataset, batch_size=1000, shuffle=False)

- i- Image batch shape is `[batch_size, C, H, W]`
  - For CIFAR-10: `[batch_size, 3, 32, 32]`
- Label batch shape is `[batch_size]`

In [23]:
device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)
print(f"Device: {device}")

Device: mps


In [40]:
class ConvBlock(nn.Module):
    def __init__(self,in_channels, out_channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(in_channels=in_channels,out_channels=out_channels,kernel_size=3,padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2,stride=2),
        )

    def forward(self, x):
        return self.block(x)


In [54]:
class CIFAR_CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            ConvBlock(in_channels=3, out_channels=32),
            ConvBlock(in_channels=32, out_channels=64),
            ConvBlock(in_channels=64, out_channels=128),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 512), #(intput features, neurons)
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512,10)
        )

    def forward(self,x):
        x = self.features(x)
        x = self.classifier(x)
        return x




In [55]:
import torch.optim.lr_scheduler as lr_scheduler
model = CIFAR_CNN().to(device)




In [71]:
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005, weight_decay=0.0001)
scheduler  = lr_scheduler.ReduceLROnPlateau(optimizer, mode='max',
                                            factor=0.5,patience=2,
                                            min_lr=1e-6)

In [72]:
#DEBUGGING
# for param in model.parameters():
#     print(param.shape)

# params = sum(param.numel() for param in model.parameters())
# print(params)
#
# for name,param in model.named_parameters():
#     print(f'{name} : {param.shape}')
#printing modules
# for module in model.modules():
#     print(module)

In [73]:
def train_model(model, train_loader,loss_function,optimizer,device=device,):
    model.train() # SET MODE TO TRAIN
    running_loss = 0.0
    for images,labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs=model(images)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()

        #Tracking progress
        running_loss += loss.item()

    # AFTER all batches are finished
    avg_loss = running_loss / len(train_loader)
    print(f"Loss: {avg_loss:.2f}")






In [74]:
# def validation_model(model, validation_loader, loss_function,device=device):
#     val_result = {}
#     model.eval() # SET MODE TO EVAL
#     with torch.no_grad():
#         for inputs,labels in validation_loader:
#             inputs,labels = inputs.to(device), labels.to(device)
#             outputs = model(inputs)
#             evaluate_metrics(model,validation_loader,device)
#             loss = loss_function(outputs, labels)
#
#


In [75]:
from torchmetrics.classification import  MulticlassAccuracy,MulticlassPrecision, MulticlassRecall, MulticlassF1Score

def evaluate_metrics(model,dataloader, device, num_classes=10):
    model.eval()

    metrics = {
        'accuracy': MulticlassAccuracy(num_classes=num_classes, average='micro').to(device),
        'precision': MulticlassPrecision(num_classes=num_classes, average='macro').to(device),
        'recall': MulticlassRecall(num_classes=num_classes, average='macro').to(device),
        'f1': MulticlassF1Score(num_classes=num_classes, average='macro').to(device)
    }

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)

            for metric in metrics.values():
                metric.update(outputs, labels)
    return{
        name: metric.compute()
        for name, metric in metrics.items()
    }


In [77]:
num_epochs = 20

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}")
    train_model(model, train_loader, loss_function, optimizer)
    result =evaluate_metrics(model, val_loader, device)

    scheduler.step(result['accuracy'].item())
    print(f"Learning rate: {optimizer.param_groups[0]['lr']:.8f}")
    for name, score in result.items():
        print(f"{name}: {score.item() * 100:.2f}%")



Epoch 1
Loss: 0.54
Learning rate: 0.00050000
accuracy: 82.18%
precision: 82.49%
recall: 82.07%
f1: 82.06%

Epoch 2
Loss: 0.53
Learning rate: 0.00050000
accuracy: 82.62%
precision: 83.04%
recall: 82.60%
f1: 82.68%

Epoch 3
Loss: 0.52
Learning rate: 0.00050000
accuracy: 82.38%
precision: 82.99%
recall: 82.24%
f1: 82.39%

Epoch 4
Loss: 0.52
Learning rate: 0.00050000
accuracy: 82.86%
precision: 83.34%
recall: 82.70%
f1: 82.64%

Epoch 5
Loss: 0.51
Learning rate: 0.00050000
accuracy: 83.34%
precision: 83.54%
recall: 83.23%
f1: 83.23%

Epoch 6
Loss: 0.50
Learning rate: 0.00050000
accuracy: 82.86%
precision: 83.62%
recall: 82.70%
f1: 82.85%

Epoch 7
Loss: 0.50
Learning rate: 0.00050000
accuracy: 83.70%
precision: 83.72%
recall: 83.59%
f1: 83.49%

Epoch 8
Loss: 0.49
Learning rate: 0.00050000
accuracy: 83.44%
precision: 83.63%
recall: 83.32%
f1: 83.37%

Epoch 9
Loss: 0.49
Learning rate: 0.00050000
accuracy: 81.66%
precision: 82.57%
recall: 81.64%
f1: 81.32%

Epoch 10
Loss: 0.48
Learning rate: 0

In [78]:
result = evaluate_metrics(model,test_loader,device)
for name, score in result.items():
    print(f"{name}: {score.item() * 100:.2f}%")

accuracy: 85.69%
precision: 85.69%
recall: 85.69%
f1: 85.65%


In [79]:
import json
from pathlib import Path
save_dir= Path("saved_models")

save_dir = Path("/Users/jimmylawson/Desktop/DL lessons/pyTorchlesson/CIFAR10-CNN")

#save the current model's learned weights
torch.save(model.state_dict(), save_dir / "cifar10_cnn_weights.pth")

#Record test metrics as percentages
test_result = {
    name: score.item() * 100 for name,score in result.items()
}

with open(save_dir / "test_results.json", "w") as file:
    json.dump(test_result, file, indent=4)

print(f"Saved weights and test results to: {save_dir.resolve()}")


Saved weights and test results to: /Users/jimmylawson/Desktop/DL lessons/pyTorchlesson/CIFAR10-CNN


In [80]:
weights = torch.load(
    save_dir / "cifar10_cnn_weights.pth",
    map_location=device,
    weights_only=True
)

for name, tensor in weights.items():
    print(f"{name}: {tensor.shape}")

features.0.block.0.weight: torch.Size([32, 3, 3, 3])
features.0.block.0.bias: torch.Size([32])
features.0.block.1.weight: torch.Size([32])
features.0.block.1.bias: torch.Size([32])
features.0.block.1.running_mean: torch.Size([32])
features.0.block.1.running_var: torch.Size([32])
features.0.block.1.num_batches_tracked: torch.Size([])
features.1.block.0.weight: torch.Size([64, 32, 3, 3])
features.1.block.0.bias: torch.Size([64])
features.1.block.1.weight: torch.Size([64])
features.1.block.1.bias: torch.Size([64])
features.1.block.1.running_mean: torch.Size([64])
features.1.block.1.running_var: torch.Size([64])
features.1.block.1.num_batches_tracked: torch.Size([])
features.2.block.0.weight: torch.Size([128, 64, 3, 3])
features.2.block.0.bias: torch.Size([128])
features.2.block.1.weight: torch.Size([128])
features.2.block.1.bias: torch.Size([128])
features.2.block.1.running_mean: torch.Size([128])
features.2.block.1.running_var: torch.Size([128])
features.2.block.1.num_batches_tracked: to